# Adım 4: Veri Bölme, Ölçeklendirme ve Model Eğitimleri
Bu son aşamada veri bölünerek ölçeklendirilecek, tüm algoritmalar eğitilecek, metrik raporları basılacak ve ROC-AUC eğrisi kaydedilecektir.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_squared_error, log_loss, roc_curve

print("4/5: Veri bölünüyor ve ölçeklendiriliyor...")
df_temiz = pd.read_csv('../data/analize_hazir_veri.csv')

y = df_temiz['ASTHMA3']
X = df_temiz.drop('ASTHMA3', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print("5/5: Farklı modeller eğitiliyor ve sonuçlar hesaplanıyor...\n")

modeller = {
    "Lojistik Regresyon (Baseline)": LogisticRegression(),
    "Karar Ağacı (Baseline)": DecisionTreeClassifier(random_state=42),
    "Random Forest (Gelişmiş)": RandomForestClassifier(random_state=42),
    "Yapay Sinir Ağı (MLP)": MLPClassifier(max_iter=1000, random_state=42),
    "KNN (Optimize Edilmiş)": KNeighborsClassifier(n_neighbors=31, weights='distance')
}

plt.figure(figsize=(10, 8))

for isim, model in modeller.items():
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    roc = roc_auc_score(y_test, y_prob)
    mse = mean_squared_error(y_test, y_pred)
    ll = log_loss(y_test, y_prob)
    
    print(f"--- {isim} ---")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc:.4f}")
    print(f"MSE (Hata): {mse:.4f} | Log Loss: {ll:.4f}\n")
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{isim} (AUC = {roc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', label='Rastgele Guess (Yazı-Tura)')
plt.xlabel('False Positive Rate (Yalancı Pozitif Oranı)')
plt.ylabel('True Positive Rate (Doğru Pozitif Oranı)')
plt.title('Modellerin ROC-AUC Eğrileri Karşılaştırması')
plt.legend(loc='lower right')

# ROC eğrisi grafiğini otomatik olarak analysis klasörüne kaydediyoruz
plt.savefig('../analysis/roc_auc_egrisi.png', dpi=300, bbox_inches='tight')
print("ROC Eğrisi 'analysis/roc_auc_egrisi.png' olarak başarıyla kaydedildi.")
plt.show()